In [ ]:
from langchain_core.prompts import PromptTemplate


username = "Harsh"
topic = "Finanacial Crisis"

# for single string prompts.
prompt = "You are a helpful Chat Assistant. Hello" + username + "Explain: " + topic

# this is very hard to version control and validate the input variables. Not suggested for Production.
# these don't use prompt templates.


## USING PROMPT TEMPLATES.
# this allows you to dynamically assign keywords in a prompt.
prompt = PromptTemplate(
    input_variables = ["username", "topic"],
    template= """You have to act like an expert Financial Analyst.
    Greet {username} with a sweet hello and then 
    explain {topic} with clear cut examples.
    """
)

formatted_prompt = prompt.invoke(
    {
        "username" : username,
        "topic": topic
    }
)

print(type(formatted_prompt))

<class 'langchain_core.prompt_values.StringPromptValue'>


In [ ]:
# a little easier way to define simple string prompts.

prompt = PromptTemplate.from_template(
    template = """
    You have to act like an expert Financial Analyst.
    Greet {username} with a sweet hello and then 
    explain {topic} with clear cut examples.
"""
)

formatted_prompt = prompt.invoke(
    {
        "username" : username,
        "topic": topic
    }
)

# Same output as the previous one.
print(formatted_prompt)

text='\n    You have to act like an expert Financial Analyst.\n    Greet Harsh with a sweet hello and then \n    explain Finanacial Crisis with clear cut examples.\n'


In [ ]:
# modern Chat models don't expect a flat string as the input.
# They need structured prompts which consists of system, human and assistant prompts
# all included in a single body. 
# For this purpose we use ChatPromptTemplate.

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

prompt = ChatPromptTemplate.from_messages(
    messages = [
        ("system", "You have to act like an Expert Surgeon who is also very accomplished in AI. You try to explain everything in terms of Biology."),
        ("human", "Please explain the {topic}. Give clear 2-3 real world examples on the topic.")
    ]
)

# or 
prompt = ChatPromptTemplate(
    messages = [
        ("system", "You have to act like an Expert Surgeon who is also very accomplished in AI. You try to explain everything in terms of Biology."),
        ("human", "Please explain the {topic}. Give clear 2-3 real world examples on the topic.")
    ]
)
# or 
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You have to act like an Expert Surgeon who is also very accomplished in AI. You try to explain everything in terms of Biology."),
        ("human", "Please explain the {topic}. Give clear 2-3 real world examples on the topic."),
        ("ai", "Hi {username}.") # for context of the user's name.
    ]
)

# the above 3 give the same output. 
formatted_prompt = prompt.invoke({"topic" : "Fomation of Rust.", "username": username})
print(formatted_prompt)

messages=[SystemMessage(content='You have to act like an Expert Surgeon who is also very accomplished in AI. You try to explain everything in terms of Biology.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Please explain the Fomation of Rust.. Give clear 2-3 real world examples on the topic.', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi Harsh.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [27]:
# your LLM can't store the context for you. 
# Hence, each LLM call is stateless and is like making a new request everytime.
# Therefore, you need to provide your LLM with context.

# for this purpose you can inject messages in your ChatPromptTemplate to give your LLM more context.

from langchain_core.prompts import MessagesPlaceholder
# you use MessagePlaceholder to inject a list of messages within your prompt.

# Prompt to be given to the LLM.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You have to act like an Expert Surgeon who is also very accomplished in AI. You try to explain everything in terms of Biology."),
        MessagesPlaceholder(variable_name= "message_history"),
        ("human", "{query}")
    ]
)

# Simulating a conversation
chat_history = [
    HumanMessage(content="I'm learning Python"),
    AIMessage(content="Great! Python is an excellent choice. What do you want to build?"),
]

formatted_prompt = prompt.invoke({
    "message_history" : chat_history,
    "query": "Explain Decorators in Python."
})

print(formatted_prompt)


messages=[SystemMessage(content='You have to act like an Expert Surgeon who is also very accomplished in AI. You try to explain everything in terms of Biology.', additional_kwargs={}, response_metadata={}), HumanMessage(content="I'm learning Python", additional_kwargs={}, response_metadata={}), AIMessage(content='Great! Python is an excellent choice. What do you want to build?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Explain Decorators in Python.', additional_kwargs={}, response_metadata={})]


In [ ]:
# Some prompt injection patterns.

# Pattern1 - Injecting retrieved context [RAG]
# Pattern2 - Few Shot Prompting
# Pattern3 - Partial Prompting Templates.


from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = "gemini-flash-latest",
    temperature = 0.8
)

chain = prompt | llm

In [ ]:
# Pattern1 - Injection Retrieved context
# Pattern 1 — Injecting retrieved context (RAG pattern preview)
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant.
Answer ONLY based on the context below.
If the answer isn't in the context, say 'I don't know.'

Context:
{context}
"""),
    ("human", "{question}")
])

# Later, context comes from your vector database
response = chain.invoke({
    "context": "LangChain is an orchestration framework...",
    "question": "What is LangChain?"
})

In [ ]:
# Pattern 2 — Few-shot prompting (giving examples inside the prompt)
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "You classify customer feedback as positive, negative, or neutral."),
    ("human", "The product is amazing!"),
    ("ai", "positive"),
    ("human", "Delivery was okay, nothing special."),
    ("ai", "neutral"),
    ("human", "Worst experience ever, broke after one day."),
    ("ai", "negative"),
    ("human", "{customer_feedback}")  # actual input
])

In [ ]:
# Pattern 3 — Partial templates
# Pre-fill some variables, leave others for later

base_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {domain} consultant."),
    ("human", "{question}")
])

# Create a specialized version with domain pre-filled
finance_prompt = base_prompt.partial(domain="financial")
legal_prompt = base_prompt.partial(domain="legal")

# Now only question needs to be provided
finance_chain = finance_prompt | llm
response = finance_chain.invoke({"question": "What is a P/E ratio?"})